# 04 --- Extraction Strategies

DocFlow supports three extraction types (text, vision, hybrid) and two modes (single, multi-agent). This notebook runs each strategy end-to-end on a real invoice PDF using Gemini Flash 2.5.

## How LLM extraction works

Once a document is parsed into text blocks, DocFlow needs to turn that unstructured content into the structured fields you defined in your Pydantic schema. This is where the LLM comes in — but there are multiple ways to present the document to the model, and each has different tradeoffs.

### Extraction types: what the LLM sees

- **Text extraction** sends the parser's text output to the LLM as tokens. This is the cheapest and fastest approach. It works well for digital PDFs where the text layer is clean and complete, but it loses all visual layout information — the LLM cannot see headers, tables, or spatial relationships.

- **Vision extraction** renders each page as an image and sends it to a vision-capable LLM. The model sees the document exactly as a human would — layout, fonts, tables, logos, stamps. This is slower and more expensive (images consume many tokens), but it excels on layout-heavy documents where spatial context matters.

- **Hybrid extraction** runs both text and vision agents in parallel, then a third "decider" agent merges the results. This gives the highest accuracy because it combines the parser's precise text with the vision model's layout understanding — at the cost of more LLM calls.

### Extraction modes: how many opinions

- **Single mode** makes one LLM call and trusts the result. Fast, cheap, usually good enough.

- **Multi-agent mode** makes N parallel calls at varied temperatures (controlling randomness), then a decider agent picks the most consistent answer. The key insight is that if 5 independent agents all agree on a value, it is much more likely to be correct than if only 1 agent produced it. This agreement ratio becomes the **trust score** — a statistical measure of extraction reliability that doesn't depend on the LLM's self-assessed confidence.

### Domain context

The `context` parameter injects domain-specific instructions into the extraction prompt. This is how you handle locale-specific formats (European decimals, date formats), industry terminology (drug names, legal clauses), or business rules (specific field conventions). It costs nothing extra — it just adds a few tokens to the prompt.

In [ ]:
import os, time

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List


class LineItem(BaseModel):
    description: str = Field(description="Line item description")
    quantity: Optional[float] = Field(default=None, description="Quantity ordered")
    unit_price: Optional[float] = Field(default=None, description="Price per unit")
    amount: float = Field(description="Line item total amount")


class Invoice(BaseModel):
    supplier_name: str = Field(description="Name of the supplier or vendor")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date the invoice was issued (YYYY-MM-DD)")
    due_date: Optional[str] = Field(default=None, description="Payment due date (YYYY-MM-DD)")
    po_number: Optional[str] = Field(default=None, description="Purchase order number")
    currency: str = Field(default="USD", description="Currency code (ISO 4217)")
    bill_to_company: Optional[str] = Field(default=None, description="Company billed")
    ship_to_company: Optional[str] = Field(default=None, description="Company shipped to")
    subtotal: Optional[float] = Field(default=None, description="Amount before tax")
    tax_rate: Optional[float] = Field(default=None, description="Tax rate as percentage")
    tax_amount: Optional[float] = Field(default=None, description="Tax amount")
    total: float = Field(description="Total amount including tax")
    payment_terms: Optional[str] = Field(default=None, description="Payment terms")
    line_items: Optional[List[LineItem]] = Field(default=None, description="Individual line items")


print(f"Invoice schema: {len(Invoice.model_fields)} fields")
for name, field in Invoice.model_fields.items():
    req = "required" if field.is_required() else "optional"
    print(f"  {name:<20} ({req})")

## Text Extraction (single agent)

The default strategy: the parser extracts text from the PDF, then a single LLM call structures it.

In [ ]:
from docflow import DocumentPipeline

text_pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    extraction_type="text",
    extraction_mode="single",
)

start = time.time()
result_text = text_pipeline.run_sync(PDF_PATH, schema=Invoice)
elapsed_text = time.time() - start

print("=== Text Extraction (single) ===")
print(f"Time: {elapsed_text:.2f}s")
print(f"Confidence: {result_text.confidence:.2f}")
print()
for key, value in result_text.data.items():
    if key != "line_items":
        print(f"  {key:<20}: {value!r}")
print(f"\n  line_items: {len(result_text.data.get('line_items', []))} items")

## Vision Extraction

Pages are rendered as images and sent directly to the vision LLM. Best for layout-heavy or graphically complex documents.

In [ ]:
vision_pipeline = DocumentPipeline(
    parser=None,
    model="gemini/gemini-2.5-flash",
    extraction_type="vision",
    vision_dpi=200,
)

start = time.time()
result_vision = vision_pipeline.run_sync(PDF_PATH, schema=Invoice)
elapsed_vision = time.time() - start

print("=== Vision Extraction ===")
print(f"Time: {elapsed_vision:.2f}s")
print(f"Confidence: {result_vision.confidence:.2f}")
print()
for key, value in result_vision.data.items():
    if key != "line_items":
        print(f"  {key:<20}: {value!r}")
print(f"\n  line_items: {len(result_vision.data.get('line_items', []))} items")

## Compare Text vs Vision

Side-by-side comparison of key fields extracted by each strategy.

In [ ]:
compare_fields = [
    "supplier_name", "invoice_number", "invoice_date", "due_date",
    "po_number", "currency", "bill_to_company", "subtotal",
    "tax_rate", "tax_amount", "total", "payment_terms",
]

print(f"{'Field':<20} {'Text':<30} {'Vision':<30} {'Match'}")
print("-" * 95)

for fname in compare_fields:
    tv = result_text.data.get(fname)
    vv = result_vision.data.get(fname)
    match = "YES" if str(tv) == str(vv) else "NO"
    print(f"  {fname:<18} {str(tv):<30} {str(vv):<30} {match}")

text_items = len(result_text.data.get("line_items") or [])
vision_items = len(result_vision.data.get("line_items") or [])
items_match = "YES" if text_items == vision_items else "NO"
print(f"  {'line_items (count)':<18} {text_items:<30} {vision_items:<30} {items_match}")

## Multi-Agent Mode

N parallel LLM calls at varied temperatures, then a decider picks the most consistent answer. Enables agreement-based trust scoring.

In [ ]:
multi_pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    extraction_type="text",
    extraction_mode="multi",
    n_instances=3,
    scoring="quantitative",
)

start = time.time()
result_multi = multi_pipeline.run_sync(PDF_PATH, schema=Invoice)
elapsed_multi = time.time() - start

print("=== Multi-Agent Extraction ===")
print(f"Time: {elapsed_multi:.2f}s")
print(f"Confidence: {result_multi.confidence:.2f}")
print()
for key, value in result_multi.data.items():
    if key != "line_items":
        print(f"  {key:<20}: {value!r}")
print(f"\n  line_items: {len(result_multi.data.get('line_items', []))} items")

In [ ]:
print("=== Per-Field Trust Scores (multi-agent) ===")
print()
print(f"{'Field':<20} {'Conf':>5}  {'Agreement':<12} {'Ratio':>6}  {'InSource':<9} {'AutoAccept'}")
print("-" * 80)

for fname, f in result_multi.fields.items():
    if fname == "line_items":
        continue
    t = f.trust
    if t:
        print(
            f"  {fname:<18} {f.confidence:>5.2f}  {t.agreement:<12} "
            f"{t.agreement_ratio:>6.2f}  {str(t.found_in_source):<9} {t.auto_accept}"
        )
    else:
        print(f"  {fname:<18} {f.confidence:>5.2f}  (no trust data)")

## Trust Score Breakdown

Detailed analysis of which fields reached full consensus across agents and which had partial agreement.

In [ ]:
full_consensus = []
partial_consensus = []
no_trust = []

for fname, f in result_multi.fields.items():
    if fname == "line_items":
        continue
    if f.trust and f.trust.agreement_ratio > 0:
        if f.trust.agreement_ratio == 1.0:
            full_consensus.append(fname)
        else:
            partial_consensus.append((fname, f.trust.agreement, f.trust.agreement_ratio))
    else:
        no_trust.append(fname)

print("=== Trust Score Breakdown ===")
print()
print(f"Full consensus ({len(full_consensus)} fields):")
for fname in full_consensus:
    t = result_multi.fields[fname].trust
    print(f"  {fname:<20} agreement={t.agreement}  ratio={t.agreement_ratio:.2f}")

print(f"\nPartial consensus ({len(partial_consensus)} fields):")
for fname, agreement, ratio in partial_consensus:
    t = result_multi.fields[fname].trust
    explanation = f"  -> {t.explanation}" if t.explanation else ""
    print(f"  {fname:<20} agreement={agreement}  ratio={ratio:.2f}{explanation}")

if no_trust:
    print(f"\nNo trust data ({len(no_trust)} fields):")
    for fname in no_trust:
        print(f"  {fname}")

auto_accepted = [n for n, f in result_multi.fields.items()
                 if n != "line_items" and f.trust and f.trust.auto_accept]
needs_review = [n for n, f in result_multi.fields.items()
                if n != "line_items" and f.trust and not f.trust.auto_accept]

print(f"\nAuto-accepted: {len(auto_accepted)} fields")
print(f"Needs review:  {len(needs_review)} fields")
if needs_review:
    print(f"  Fields requiring review: {needs_review}")

## The two confidence scores: OCR vs consensus

Beyond the legacy `trust` object, every field carries two explicit, independent scores — deliberately not blended into one number:

- **`field.ocr`** — *did we READ the page correctly?* Comes from the OCR engine's per-word recognition scores, matched back from the extracted value to the source words. Only exists when an OCR-based parser ran — a native digital PDF has nothing to mis-read, so `field.ocr` is `None` here.
- **`field.consensus`** — *did independent LLM runs INTERPRET the document the same way?* Agreement across the N instances, measured against the final value the decider chose. Only exists in multi/hybrid mode. This is the primary trust signal in vision extraction, where there may be no parser at all.

`None` always means *not applicable*, never *bad*.

Every result also reports the aggregated **token usage** across all LLM calls (instances + decider + retries), so cost is never invisible.

In [ ]:
print("=== Explicit confidence scores (multi-agent run) ===")
print()
print(f"Document OCR confidence: {result_multi.ocr}")  # None — native PDF, no OCR ran
print()
print(f"{'Field':<20} {'Consensus':<12} {'vs majority':>11}")
print("-" * 46)
for fname, f in result_multi.fields.items():
    if fname == "line_items" or f.consensus is None:
        continue
    c = f.consensus
    print(f"  {fname:<18} {c.agreement:<12} {c.majority_ratio:>11.2f}")

print()
print("=== Token usage ===")
if result_multi.usage:
    u = result_multi.usage
    print(f"  {u.total_tokens} tokens across {u.n_llm_calls} LLM calls "
          f"({u.prompt_tokens} prompt / {u.completion_tokens} completion)")
    if u.cost_usd is not None:
        print(f"  Estimated cost: ${u.cost_usd:.4f}")


## Domain Context

Passing a `context` string injects domain-specific instructions into the extraction prompt. Here we compare date formatting with and without context.

In [ ]:
context_pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    context="This is a US invoice. Amounts use standard US format (comma for thousands, period for decimals). Dates should be extracted in YYYY-MM-DD format.",
)

start = time.time()
result_ctx = context_pipeline.run_sync(PDF_PATH, schema=Invoice)
elapsed_ctx = time.time() - start

print("=== With Domain Context ===")
print(f"Time: {elapsed_ctx:.2f}s")
print(f"Confidence: {result_ctx.confidence:.2f}")
print()

print(f"{'Field':<20} {'Without Context':<25} {'With Context':<25}")
print("-" * 70)

date_fields = ["invoice_date", "due_date"]
amount_fields = ["subtotal", "tax_amount", "total"]

for fname in date_fields + amount_fields:
    no_ctx = result_text.data.get(fname)
    with_ctx = result_ctx.data.get(fname)
    print(f"  {fname:<18} {str(no_ctx):<25} {str(with_ctx):<25}")

## Summary

All four approaches compared side by side.

In [ ]:
rows = [
    ("Text (single)", result_text.confidence, elapsed_text, 1),
    ("Vision", result_vision.confidence, elapsed_vision, 1),
    ("Multi-agent (3x)", result_multi.confidence, elapsed_multi, 4),
    ("Text + context", result_ctx.confidence, elapsed_ctx, 1),
]

print("=== Extraction Strategy Comparison ===")
print()
print(f"{'Method':<22} {'Confidence':>10} {'Time (s)':>10} {'LLM Calls':>10}")
print("-" * 55)

for method, conf, elapsed, calls in rows:
    print(f"  {method:<20} {conf:>10.2f} {elapsed:>10.2f} {calls:>10}")

print()
print("Notes:")
print("  - Multi-agent uses N instances + 1 decider call")
print("  - Vision bypasses the parser entirely (reads page images)")
print("  - Context does not add LLM calls, just injects instructions")